In [ ]:
%%capture
!pip install rouge-score
!pip install accelerate -U
!pip install gradio

In [ ]:
import pandas as pd
import re

In [ ]:
data=pd.read_csv("IndianFoodDataset.csv", on_bad_lines='skip')

In [ ]:
data.head()

,Srno,RecipeName,TranslatedRecipeName,Ingredients,TranslatedIngredients,PrepTimeInMins,CookTimeInMins,TotalTimeInMins,Servings,Cuisine,Course,Diet,Instructions,TranslatedInstructions,URL
0,1,Masala Karela Recipe,Masala Karela Recipe,"6 Karela (Bitter Gourd/ Pavakkai) - deseeded,S...","6 Karela (Bitter Gourd/ Pavakkai) - deseeded,S...",15,30,45,6,Indian,Side Dish,Diabetic Friendly,"To begin making the Masala Karela Recipe,de-se...","To begin making the Masala Karela Recipe,de-se...",https://www.archanaskitchen.com/masala-karela-...
1,2,टमाटर पुलियोगरे रेसिपी - Spicy Tomato Rice (Re...,Spicy Tomato Rice (Recipe),"2-1/2 कप चावल - पका ले,3 टमाटर,3 छोटा चमच्च बी...","2-1 / 2 cups rice - cooked, 3 tomatoes, 3 teas...",5,10,15,3,South Indian Recipes,Main Course,Vegetarian,टमाटर पुलियोगरे बनाने के लिए सबसे पहले टमाटर क...,"To make tomato puliogere, first cut the tomato...",http://www.archanaskitchen.com/spicy-tomato-ri...
2,3,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,Ragi Semiya Upma Recipe - Ragi Millet Vermicel...,"1-1/2 cups Rice Vermicelli Noodles (Thin),1 On...","1-1/2 cups Rice Vermicelli Noodles (Thin),1 On...",20,30,50,4,South Indian Recipes,South Indian Breakfast,High Protein Vegetarian,"To begin making the Ragi Vermicelli Recipe, fi...","To begin making the Ragi Vermicelli Recipe, fi...",http://www.archanaskitchen.com/ragi-vermicelli...
3,4,Gongura Chicken Curry Recipe - Andhra Style Go...,Gongura Chicken Curry Recipe - Andhra Style Go...,"500 grams Chicken,2 Onion - chopped,1 Tomato -...","500 grams Chicken,2 Onion - chopped,1 Tomato -...",15,30,45,4,Andhra,Lunch,Non Vegeterian,To begin making Gongura Chicken Curry Recipe f...,To begin making Gongura Chicken Curry Recipe f...,http://www.archanaskitchen.com/gongura-chicken...
4,5,आंध्रा स्टाइल आलम पचड़ी रेसिपी - Adrak Chutney ...,Andhra Style Alam Pachadi Recipe - Adrak Chutn...,"1 बड़ा चमच्च चना दाल,1 बड़ा चमच्च सफ़ेद उरद दाल,2...","1 tablespoon chana dal, 1 tablespoon white ura...",10,20,30,4,Andhra,South Indian Breakfast,Vegetarian,आंध्रा स्टाइल आलम पचड़ी बनाने के लिए सबसे पहले ...,"To make Andhra Style Alam Pachadi, first heat ...",https://www.archanaskitchen.com/andhra-style-a...


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1186 entries, 0 to 1185
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   Srno                    1186 non-null   int64 
 1   RecipeName              1186 non-null   object
 2   TranslatedRecipeName    1186 non-null   object
 3   Ingredients             1185 non-null   object
 4   TranslatedIngredients   1185 non-null   object
 5   PrepTimeInMins          1186 non-null   int64 
 6   CookTimeInMins          1186 non-null   int64 
 7   TotalTimeInMins         1186 non-null   int64 
 8   Servings                1186 non-null   int64 
 9   Cuisine                 1186 non-null   object
 10  Course                  1186 non-null   object
 11  Diet                    1186 non-null   object
 12  Instructions            1186 non-null   object
 13  TranslatedInstructions  1186 non-null   object
 14  URL                     1186 non-null   object
dtypes: i

In [ ]:
data = data.drop(['Srno', 'RecipeName', 'Ingredients', 'Instructions'], axis=1)

In [ ]:
data.isnull().sum()

,0
TranslatedRecipeName,0
TranslatedIngredients,1
PrepTimeInMins,0
CookTimeInMins,0
TotalTimeInMins,0
Servings,0
Cuisine,0
Course,0
Diet,0
TranslatedInstructions,0


In [ ]:
data=data.dropna()

In [ ]:
def clean_text(text):

    # Remove non-alphanumeric characters and extra whitespaces
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Apply text cleaning to relevant columns
text_columns = ['TranslatedRecipeName', 'TranslatedIngredients', 'TranslatedInstructions']
for column in text_columns:
    data[column] = data[column].apply(clean_text)

In [ ]:
data.head()

,TranslatedRecipeName,TranslatedIngredients,PrepTimeInMins,CookTimeInMins,TotalTimeInMins,Servings,Cuisine,Course,Diet,TranslatedInstructions,URL
0,Masala Karela Recipe,6 Karela Bitter Gourd Pavakkai deseededSalt to...,15,30,45,6,Indian,Side Dish,Diabetic Friendly,To begin making the Masala Karela Recipedeseed...,https://www.archanaskitchen.com/masala-karela-...
1,Spicy Tomato Rice Recipe,21 2 cups rice cooked 3 tomatoes 3 teaspoons B...,5,10,15,3,South Indian Recipes,Main Course,Vegetarian,To make tomato puliogere first cut the tomatoe...,http://www.archanaskitchen.com/spicy-tomato-ri...
2,Ragi Semiya Upma Recipe Ragi Millet Vermicelli...,112 cups Rice Vermicelli Noodles Thin1 Onion s...,20,30,50,4,South Indian Recipes,South Indian Breakfast,High Protein Vegetarian,To begin making the Ragi Vermicelli Recipe fir...,http://www.archanaskitchen.com/ragi-vermicelli...
3,Gongura Chicken Curry Recipe Andhra Style Gong...,500 grams Chicken2 Onion chopped1 Tomato chopp...,15,30,45,4,Andhra,Lunch,Non Vegeterian,To begin making Gongura Chicken Curry Recipe f...,http://www.archanaskitchen.com/gongura-chicken...
4,Andhra Style Alam Pachadi Recipe Adrak Chutney...,1 tablespoon chana dal 1 tablespoon white urad...,10,20,30,4,Andhra,South Indian Breakfast,Vegetarian,To make Andhra Style Alam Pachadi first heat o...,https://www.archanaskitchen.com/andhra-style-a...


In [ ]:
data.to_csv("cleaned_IndianFoodDataset.csv", index=False)

In [ ]:
%%capture
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import GPT2LMHeadModel, GPT2Tokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments


In [ ]:
data = pd.read_csv("cleaned_IndianFoodDataset.csv")

In [ ]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)
train_data, validation_data = train_test_split(train_data, test_size=0.1, random_state=42)

In [ ]:
# Save the split datasets
train_data.to_csv("train_dataset.csv", index=False)
validation_data.to_csv("validation_dataset.csv", index=False)
test_data.to_csv("test_dataset.csv", index=False)

In [ ]:
 # Load the GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Set the pad token for the tokenizer, often needed for GPT-2 models
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load datasets from CSV files
train_dataset_raw = Dataset.from_csv("train_dataset.csv")
validation_dataset_raw = Dataset.from_csv("validation_dataset.csv")
test_dataset_raw = Dataset.from_csv("test_dataset.csv")

# Define a function to tokenize and combine relevant text columns
def preprocess_function(examples):
    # Concatenate the recipe name, ingredients, and instructions into a single text string
    # This combined text will be the input for the language model
    combined_text = [
        f"Recipe Name: {name}. Ingredients: {ing}. Instructions: {instr}"
        for name, ing, instr in zip(
            examples["TranslatedRecipeName"],
            examples["TranslatedIngredients"],
            examples["TranslatedInstructions"]
        )
    ]

    # Tokenize the combined text
    tokenized_inputs = tokenizer(
        combined_text,
        max_length=128,  # Using the original block_size
        truncation=True,
        padding="max_length" # Pad to max_length for consistent input lengths
    )
    # For causal language modeling, the labels are typically the input_ids themselves
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs

# Apply the preprocessing function to all datasets
train_dataset = train_dataset_raw.map(
    preprocess_function,
    batched=True,
    num_proc=1, # Can increase num_proc for faster processing if system resources allow
    remove_columns=train_dataset_raw.column_names # Remove original text columns to save memory
)
validation_dataset = validation_dataset_raw.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=validation_dataset_raw.column_names
)
test_dataset = test_dataset_raw.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=test_dataset_raw.column_names
)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/853 [00:00<?, ? examples/s]

Map:   0%|          | 0/95 [00:00<?, ? examples/s]

Map:   0%|          | 0/237 [00:00<?, ? examples/s]

In [ ]:

# Set up training arguments
from transformers import TrainingArguments

# Define training configuration for the model
training_args = TrainingArguments(

    output_dir="./gpt2-indian-food",
    # Folder where the trained model checkpoints and outputs will be saved

    num_train_epochs=3,
    # Number of times the model will go through the entire training dataset
    # 3 epochs = full dataset is seen 3 times

    per_device_train_batch_size=2,
    # Number of training samples processed at once per device (CPU/GPU)
    # Small batch size is often used when GPU memory is limited

    save_steps=10_000,
    # Save a checkpoint every 10,000 training steps
    # A step = one batch forward + backward pass

    save_total_limit=2,
    # Keep only the latest 2 saved checkpoints
    # Older checkpoints will be deleted automatically to save disk space

    prediction_loss_only=True
    # During evaluation, only compute and return the loss
    # Speeds up evaluation by not storing full predictions
)


In [ ]:
# from transformers import Trainer, DataCollatorForLanguageModeling

# Set up the Trainer object
trainer = Trainer(

    model=model,
    # The pre-trained GPT-2 model (or your fine-tuned version)
    # This is the model that will be trained

    args=training_args,
    # The TrainingArguments object we defined earlier
    # Contains all training configuration like epochs, batch size, etc.

    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        # Tokenizer used to convert text into token IDs

        mlm=False
        # mlm = Masked Language Modeling
        # False because GPT-2 is a Causal Language Model (CLM)
        # It predicts the next word, not masked words like BERT
    ),

    train_dataset=train_dataset,
    # Dataset used for training the model

    eval_dataset=validation_dataset,
    # Dataset used for evaluation during training
)


In [ ]:
#start training
trainer.train()

Step,Training Loss
500,2.589160
1000,2.064916


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1281, training_loss=2.245332930815769, metrics={'train_runtime': 157.3967, 'train_samples_per_second': 16.258, 'train_steps_per_second': 8.139, 'total_flos': 167161577472000.0, 'train_loss': 2.245332930815769, 'epoch': 3.0})

In [ ]:
#save the model
trainer.save_model("gpt2-indian-food")

# Save the configuration file
model = trainer.model
model.config.save_pretrained("gpt2-indian-food_config.json")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Evaluate on the test set
results = trainer.evaluate(test_dataset)
print(results)


{'eval_loss': 2.036993980407715, 'eval_runtime': 2.7031, 'eval_samples_per_second': 87.678, 'eval_steps_per_second': 11.098, 'epoch': 3.0}


In [ ]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

# Download NLTK resources
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
# Load the trained model and tokenizer
model = GPT2LMHeadModel.from_pretrained("gpt2-indian-food")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Function to generate text using the model
def generate_text(prompt, max_length=100, temperature=1.0):
    """
    Generate text from a trained GPT-2 model.

    Parameters:
    prompt (str): The starting text for generation
    max_length (int): Maximum total length of generated sequence (including prompt)
    temperature (float): Controls randomness of predictions
    """

    # Convert input text into token IDs (numbers the model understands)
    # return_tensors="pt" returns PyTorch tensors
    input_ids = tokenizer.encode(prompt, return_tensors="pt")

    # Generate output tokens using the model
    output = model.generate(
        input_ids,
        max_length=max_length,              # Maximum total length of output sequence
        temperature=temperature,            # Controls randomness (higher = more creative)
        num_beams=5,                        # Beam search with 5 beams (more structured output)
        no_repeat_ngram_size=2              # Prevents repeating 2-word sequences
    )

    # Convert generated token IDs back into readable text
    generated_text = tokenizer.decode(
        output[0],                          # Take first generated sequence
        skip_special_tokens=True            # Remove special tokens like <EOS>
    )

    return generated_text

# Example prompts and references
prompts = ["To make Masala Karela,",
           "Tomato Puliogere is prepared by",
           "Ragi Vermicelli Recipe starts with",
           "Gongura Chicken Curry Recipe first prep all the ingredients",
           "Andhra Style Alam Pachadi is made by"]

references = ["To begin making the Masala Karela Recipe, de-seed the karela and slice.",
              "To make tomato puliogere, first cut the tomatoes.",
              "To begin making the Ragi Vermicelli Recipe, first steam the ragi vermicelli.",
              "To begin making Gongura Chicken Curry Recipe, first prep all the ingredients.",
              "To make Andhra Style Alam Pachadi, first heat oil in a pan."]

# Generate text and calculate BLEU and ROUGE scores

smooth = SmoothingFunction().method1
# Define smoothing function for BLEU
# Smoothing is needed because BLEU can become 0
# if higher-order n-grams don't match at all

bleu_scores = []
# List to store BLEU scores for each generated example

rouge_scores = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)
# Initialize ROUGE scorer
# rouge1 → unigram overlap (word-level overlap)
# rouge2 → bigram overlap (two-word phrases)
# rougeL → longest common subsequence similarity
# use_stemmer=True allows matching "eat" and "eating"

for prompt, reference in zip(prompts, references):
    # Loop through each prompt and its true reference text

    generated_text = generate_text(prompt)
    # Generate text from your GPT-2 model

    # ---------------- BLEU Score ----------------

    bleu_score = sentence_bleu(
        [reference.split()],              # Reference text (must be inside list)
        generated_text.split(),           # Generated text tokens
        smoothing_function=smooth         # Apply smoothing
    )

    bleu_scores.append(bleu_score)
    # Store BLEU score for later averaging

    print(f"Prompt: {prompt}")
    print(f"Generated: {generated_text}")
    print(f"Reference: {reference}")
    print(f"BLEU Score: {bleu_score:.4f}\n")

    # ---------------- ROUGE Score ----------------

    rouge_score = rouge_scores.score(
        generated_text,   # Hypothesis
        reference         # Ground truth
    )

    print(f"ROUGE Scores: {rouge_score}\n")

# ---------------- Average BLEU ----------------

average_bleu = sum(bleu_scores) / len(bleu_scores)
# Compute average BLEU across all examples

print(f"Average BLEU Score: {average_bleu:.4f}")



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Prompt: To make Masala Karela,
Generated: To make Masala Karela, we will first make the masala karela recipe. We will make it in a pressure cooker with 2 cups of water and 2 tablespoons of oil. Now heat a heavy bottomed pan on medium heat and add the lentils and cook until they turn golden brown and crisp Turn off the flame and let it cook for 10 to 15 minutes After 10 minutes remove the lid and allow it to cool down Once it cools down take it out of the cooker and keep
Reference: To begin making the Masala Karela Recipe, de-seed the karela and slice.
BLEU Score: 0.0032

ROUGE Scores: {'rouge1': Score(precision=0.6923076923076923, recall=0.10112359550561797, fmeasure=0.17647058823529413), 'rouge2': Score(precision=0.3333333333333333, recall=0.045454545454545456, fmeasure=0.08), 'rougeL': Score(precision=0.6153846153846154, recall=0.0898876404494382, fmeasure=0.1568627450980392)}

Prompt: Tomato Puliogere is prepared by
Generated: Tomato Puliogere is prepared by the same recipe. Ingredi